# Compare index sets

Discover benchmark experiments with **Yanex**, inspect their metrics, then use
**pandas** and **matplotlib** to investigate individual queries.

Run the benchmark sweep first. Select this project's `.venv` kernel in VS Code
and run the cells in order. This notebook reads saved results; MongoDB can be stopped.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import yanex.results as yr
from IPython.display import display

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## 1. Discover experiments

`get_experiments()` searches your local Yanex store. Always specify the project:
it does not infer one from the notebook's directory. Add `tags=["my-comparison"]`,
`name="3-benchmark*"`, or `started_after="2026-10-01"` to narrow the search.
We list every status first so failed runs are visible.

In [ ]:
experiments = yr.get_experiments(
    project="bondi-demo-project",
    script_pattern="benchmark.py",
    sort_by="created_at",
    sort_desc=True,
)
assert experiments, "Run the benchmark sweep before opening this notebook."

runs = pd.DataFrame(
    [
        {
            "experiment_id": exp.id,
            "name": exp.name,
            "status": exp.status,
            "index_set": exp.get_param("index_set"),
            "workload_id": exp.get_dependency("workload").id,
        }
        for exp in experiments
    ]
)
display(runs)

## 2. Select a comparison

Use the newest completed run's workload, then the newest completed run for each
index set on that **same workload**. This selects by creation time, not speed.
To revisit an older workload, set `workload_id` from the table above. For a specific
sweep, narrow the discovery filters rather than pooling unrelated runs.

In [ ]:
completed = runs[runs["status"] == "completed"]
assert not completed.empty, "No completed benchmark runs found."
workload_id = completed.iloc[0]["workload_id"]
index_sets = ["baseline", "single", "compound"]

selected = (
    completed[completed["workload_id"] == workload_id]
    .drop_duplicates("index_set", keep="first")
    .set_index("index_set")
)
assert set(index_sets) <= set(selected.index), "Complete all three index sets for this workload."
selected = selected.loc[index_sets]
by_id = {exp.id: exp for exp in experiments}
benchmarks = [by_id[experiment_id] for experiment_id in selected["experiment_id"]]
# Compare equivalent measurement settings, not smoke tests with different repetition counts.
assert all(
    len({exp.get_param(key) for exp in benchmarks}) == 1
    for key in ["warmups", "repetitions", "timeout_seconds"]
), "Choose runs with matching measurement settings."
display(selected)

## 3. Read logged metrics

`exp.get_metrics()` returns a DataFrame of the values logged by one experiment.
Our benchmark logs one row containing the query count and median workload time.
These are client-observed times, including fetching result rows.

In [ ]:
exp = benchmarks[0]
display(exp.get_metrics())

summary = (
    pd.concat(
        [
            exp.get_metrics().assign(index_set=exp.get_param("index_set"), experiment_id=exp.id)
            for exp in benchmarks
        ],
        ignore_index=True,
    )
    .set_index("index_set")
    .loc[index_sets]
)
summary["speedup_vs_baseline"] = (
    summary.loc["baseline", "median_workload_ms"] / summary["median_workload_ms"]
)
display(
    summary[["experiment_id", "query_count", "median_workload_ms", "speedup_vs_baseline"]].round(2)
)

## 4. Plot metrics with Yanex

`yr.get_metrics()` combines selected experiments into the long-form DataFrame
expected by `yr.plot_metrics()`. A single logged value becomes a bar; metrics
logged over multiple steps would become curves. Speedup above 1 means faster
than baseline. This chart contains summaries, not the individual repetitions.

In [ ]:
metrics = yr.get_metrics(
    ids=[exp.id for exp in benchmarks],
    metrics=["median_workload_ms"],
    params=["index_set"],
)
display(metrics)
fig, axes = yr.plot_metrics(metrics, group_by="index_set", sort_by="value")
axes[0].set_ylabel("Median workload time (ms; lower is better)")
plt.show()

## 5. Load individual query measurements

The artifact has one row per query per measured repetition. Add the experiment
identity and index set before combining the rows. `query_id` refers to the saved
workload order; `pattern_id` groups queries with the same shape.

In [ ]:
queries = pd.concat(
    [
        pd.DataFrame(exp.load_artifact("query_results.jsonl")).assign(
            index_set=exp.get_param("index_set"),
            experiment_id=exp.id,
        )
        for exp in benchmarks
    ],
    ignore_index=True,
)
assert queries["status"].eq("success").all(), "Inspect failed queries before comparing timings."
repetitions = int(benchmarks[0].get_param("repetitions"))
assert queries.groupby(["index_set", "query_id"]).size().eq(repetitions).all()
assert not queries.duplicated(["index_set", "query_id", "repetition"]).any()
display(queries.head())

stats = queries.groupby(["pattern_id", "query_id", "index_set"])["elapsed_ms"].agg(
    count="count",
    median_ms="median",
    mean_ms="mean",
    min_ms="min",
    max_ms="max",
)
display(stats.round(2))

## 6. Compare every query, grouped by pattern

Each pattern gets two plots: median query times and box plots of the measured
repetitions. The dots show the actual observations (three per box with the default
config). A box based on three observations describes these runs; it is not strong
evidence about the underlying timing distribution.

Change `pattern_ids` to a shorter list, such as `[0]`, to focus on one pattern.
All charts use milliseconds, but vertical scales differ between patterns.

In [ ]:
pattern_ids = sorted(queries["pattern_id"].unique())
colors = dict(zip(index_sets, plt.get_cmap("tab10").colors))

for pattern_id in pattern_ids:
    subset = queries[queries["pattern_id"] == pattern_id]
    medians = subset.pivot_table(
        index="query_id",
        columns="index_set",
        values="elapsed_ms",
        aggfunc="median",
    ).reindex(columns=index_sets)
    fig, (bar_ax, box_ax) = plt.subplots(2, 1, figsize=(14, 8), constrained_layout=True)
    medians.plot.bar(ax=bar_ax, color=[colors[name] for name in index_sets], rot=0)
    bar_ax.set(
        title=f"Pattern {pattern_id}: median query times", ylabel="Time (ms)", xlabel="Query ID"
    )

    # Offset each index set slightly so the same query's boxes sit together.
    for offset, index_set in enumerate(index_sets):
        values = [
            subset[(subset["query_id"] == query_id) & (subset["index_set"] == index_set)][
                "elapsed_ms"
            ]
            for query_id in medians.index
        ]
        positions = [
            i + (offset - (len(index_sets) - 1) / 2) * (0.8 / len(index_sets))
            for i in range(len(values))
        ]
        boxes = box_ax.boxplot(
            values,
            positions=positions,
            widths=0.6 / len(index_sets),
            patch_artist=True,
            manage_ticks=False,
        )
        for box in boxes["boxes"]:
            box.set_facecolor(colors[index_set])
            box.set_alpha(0.5)
        for position, observations in zip(positions, values):
            box_ax.scatter(
                [position] * len(observations), observations, color=colors[index_set], s=14
            )
    box_ax.set_xticks(range(len(medians)), medians.index)
    box_ax.set(
        title=f"Pattern {pattern_id}: all {repetitions} repetitions",
        ylabel="Time (ms)",
        xlabel="Query ID",
    )
    plt.show()

## 7. Connect timings to query shapes

Load the native BONDI patterns from the workload experiment to interpret the charts.
Does an index help all instances of a pattern? Which fields would you use for set C?
Remember that changed index sets, query limits, returned row counts and cache state
can all influence execution time.

In [ ]:
from bondi.mongodb import MongoQueryPattern

workload_experiment = benchmarks[0].get_dependency("workload")
patterns = [
    MongoQueryPattern.from_dict(row) for row in workload_experiment.load_artifact("patterns.jsonl")
]
for pattern_id, pattern in enumerate(patterns):
    print(f"Pattern {pattern_id}: {pattern}")

## Next steps

- Add set C, rerun the benchmark against the **same workload**, and include `"C"` in `index_sets`.
- Repeat the comparison in a different execution order to investigate cache effects.
- Use `stats.to_csv(...)` or `fig.savefig(...)` to export tables and figures for a report.

References: [Yanex](https://github.com/rueckstiess/yanex),
[pandas tutorials](https://pandas.pydata.org/docs/getting_started/intro_tutorials/),
[matplotlib tutorials](https://matplotlib.org/stable/tutorials/index.html).